# 01 — Exploratory data analysis

Dataset: Kaggle [AI vs Human Text](https://www.kaggle.com/datasets/shanegerami/ai-vs-human-text) (`text`, `generated`). After cleaning, `generated` is stored as `label` with **0 = human** and **1 = AI**.

This notebook checks whether the task is a real writing-style problem or something a model can solve from artifacts: the class base rate, essay length, repeated prompts, and duplicate strings. Plots are written to `outputs/figures/`. The modeling sample is a balanced draw of at most 20,000 rows (seed 42); the distribution plots below use the full cleaned corpus so the original base rate and length gap stay visible.


In [ ]:
%matplotlib inline

import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "config.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find src/config.py. Open this notebook from the ai-text-detector project."
    )

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import FIGURES_DIR, MAX_TRAIN_SAMPLES, SEED
from src.data_utils import (
    _select_text_and_label,
    clean_dataframe,
    load_raw_dataset,
    make_splits,
    set_seed,
    subsample_balanced,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
set_seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CLASS_COLORS = {"Human": "#4C78A8", "AI": "#F58518"}
CLASS_NAMES = {0: "Human", 1: "AI"}


def save_figure(fig: plt.Figure, filename: str) -> Path:
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"saved {path.relative_to(ROOT).as_posix()}")
    return path


## Load

`load_raw_dataset` reads `data/raw/AI_Human.csv` and prints the columns it actually found. Cleaning then maps the text and label fields onto `text` and `label` if the header differs.


In [ ]:
raw = load_raw_dataset()
print("raw shape:", raw.shape)
print("raw columns:", list(raw.columns))
raw.head(3)


## Clean

Null and blank rows are dropped, whitespace is stripped, duplicate strings are removed, and texts under 20 words are discarded. The same string with both labels is reported before the extra copy is dropped.


In [ ]:
prepared = _select_text_and_label(raw)
n_conflicts = int((prepared.groupby("text")["label"].nunique() > 1).sum())
cleaned = clean_dataframe(raw)
cleaned = cleaned.copy()
cleaned["n_words"] = cleaned["text"].str.split().str.len()
cleaned["source"] = cleaned["label"].map(CLASS_NAMES)

print(f"raw rows: {len(raw):,}")
print(f"rows after null/blank drop: {len(prepared):,}")
print(f"texts that appear with both labels: {n_conflicts:,}")
print(f"cleaned rows: {len(cleaned):,}")
print(cleaned["source"].value_counts().rename("count").to_frame())


## Class balance


In [ ]:
counts = cleaned["source"].value_counts().reindex(["Human", "AI"])
shares = counts / counts.sum()

fig, ax = plt.subplots(figsize=(6.2, 4.2))
sns.barplot(
    x=list(counts.index),
    y=counts.to_numpy(),
    hue=list(counts.index),
    palette=CLASS_COLORS,
    legend=False,
    ax=ax,
)
for bar, count, share in zip(ax.patches, counts.to_numpy(), shares.to_numpy()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{int(count):,}\n({share:.1%})",
        ha="center",
        va="bottom",
    )
ax.set_xlabel("")
ax.set_ylabel("Texts")
ax.set_title("Class balance after cleaning")
ax.set_ylim(0, counts.max() * 1.18)
fig.tight_layout()
save_figure(fig, "class_balance.png")
plt.show()


These counts are the cleaned corpus, before the 50/50 modeling sample. The training pipeline then draws equal classes, so this chart is the only view of the original base rate. If one class dominates here, accuracy on a balanced test set will look better than accuracy on the natural mix; report F1 and a majority-class baseline beside it.


## Text length


In [ ]:
length_stats = (
    cleaned.groupby("source")["n_words"]
    .agg(count="size", mean="mean", median="median", std="std")
    .reindex(["Human", "AI"])
    .round(1)
)
print(length_stats)

xmax = float(cleaned["n_words"].quantile(0.99))
fig, ax = plt.subplots(figsize=(8.2, 4.6))
sns.histplot(
    data=cleaned,
    x="n_words",
    hue="source",
    hue_order=["Human", "AI"],
    stat="density",
    common_norm=False,
    bins=60,
    element="step",
    fill=True,
    alpha=0.35,
    palette=CLASS_COLORS,
    ax=ax,
)
ax.set_xlim(0, xmax)
ax.set_xlabel("Words")
ax.set_ylabel("Density")
ax.set_title("Word count by class (axis clipped at the 99th percentile)")
fig.tight_layout()
save_figure(fig, "length_distribution.png")
plt.show()


Each curve is normalized on its own, so the taller class does not hide the other. The axis stops at the 99th percentile only for display; the printed mean and median use every cleaned text. A gap of a few dozen words is a weak cue. Distributions that barely overlap mean a later model can score well by counting tokens, which is why the length-only baseline at the end of this notebook is fit on the training split and scored on the test split.


## Top 20 words per class

Stopwords are removed so the lists show content words rather than "the" and "and". Each class uses a seeded sample of at most 20,000 texts.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

TOP_WORD_DOCS = 20_000


def top_words(texts: pd.Series, n: int = 20) -> tuple[pd.DataFrame, int]:
    sample = texts
    if len(texts) > TOP_WORD_DOCS:
        sample = texts.sample(TOP_WORD_DOCS, random_state=SEED)
    vectorizer = CountVectorizer(
        stop_words="english",
        token_pattern=r"[a-zA-Z]{2,}",
        lowercase=True,
        max_features=5000,
    )
    matrix = vectorizer.fit_transform(sample)
    totals = np.asarray(matrix.sum(axis=0)).ravel()
    vocabulary = np.asarray(vectorizer.get_feature_names_out())
    order = np.argsort(totals)[::-1][:n]
    table = pd.DataFrame({"word": vocabulary[order], "count": totals[order]})
    return table, int(len(sample))


human_words, human_n = top_words(cleaned.loc[cleaned["label"] == 0, "text"])
ai_words, ai_n = top_words(cleaned.loc[cleaned["label"] == 1, "text"])
print("Human sample:", f"{human_n:,}")
print(human_words.to_string(index=False))
print("\nAI sample:", f"{ai_n:,}")
print(ai_words.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 6.2))
for ax, name, table, sample_size in (
    (axes[0], "Human", human_words, human_n),
    (axes[1], "AI", ai_words, ai_n),
):
    ax.barh(table["word"][::-1], table["count"][::-1], color=CLASS_COLORS[name])
    ax.set_title(f"{name} top 20 (n={sample_size:,})")
    ax.set_xlabel("Count in sample")
fig.suptitle("Most frequent content words", y=1.02)
fig.tight_layout()
save_figure(fig, "top_words_per_class.png")
plt.show()


Shared topical words (school essays, cars, elections, and so on) mean the two classes were often written to the same prompts, so a bag-of-words model can pick up the topic mix instead of whether a person wrote the text. Words that show up on only one side are worth reading in the samples below: a generation tic such as "delve" or "tapestry" is a real cue, while a topic that appears in only one class is a confound. Function words were removed here on purpose, so similar lists do not prove the styles match.


## Sample texts

Three texts per class, chosen with seed 42. Display is cut at 120 words so the notebook stays readable; the word count is the full text.


In [ ]:
parts = [
    group.sample(n=min(3, len(group)), random_state=SEED)
    for _, group in cleaned.groupby("label")
]
samples = pd.concat(parts, ignore_index=True)

for _, row in samples.iterrows():
    words = str(row["text"]).split()
    preview = " ".join(words[:120])
    if len(words) > 120:
        preview += " ..."
    print(f"\n[{CLASS_NAMES[int(row['label'])]} | {len(words)} words]")
    print(preview)


Read the excerpts for leftover prompt headers, "as an AI" disclaimers, and a house style that is much smoother than the human essays. Those are dataset artifacts. A detector that keys off them can fail as soon as the generator or the essay prompt changes, which is the cross-domain question this project is built to test.


## Leakage and length shortcuts

Exact duplicate strings cannot fall into two splits, because cleaning drops them first. That does not cover a shared assignment prompt. The checks below measure (1) strings that were labeled both human and AI, (2) 15-word openings that the test set still shares with train, and (3) a length threshold chosen on train only and scored on test.

This section also writes `data/processed/train.csv`, `val.csv`, and `test.csv` through `make_splits`, so the leakage check is the same split the models will train on.


In [ ]:
modeling = subsample_balanced(cleaned.drop(columns=["source"]), MAX_TRAIN_SAMPLES, seed=SEED)
train, val, test = make_splits(modeling)


def opening(text: str, n_tokens: int = 15) -> str:
    return " ".join(str(text).lower().split()[:n_tokens])


train_texts = set(train["text"])
val_texts = set(val["text"])
val_exact = int(val["text"].isin(train_texts).sum())
test_exact = int(test["text"].isin(train_texts).sum())
val_test_exact = int(test["text"].isin(val_texts).sum())

human_openings = set(cleaned.loc[cleaned["label"] == 0, "text"].map(opening))
ai_prefix_rate = float(cleaned.loc[cleaned["label"] == 1, "text"].map(opening).isin(human_openings).mean())
train_openings = set(train["text"].map(opening))
test_prefix_rate = float(test["text"].map(opening).isin(train_openings).mean())

train_words = train["text"].str.split().str.len().to_numpy()
test_words = test["text"].str.split().str.len().to_numpy()
y_train = train["label"].to_numpy()
y_test = test["label"].to_numpy()
thresholds = np.unique(np.quantile(train_words, np.linspace(0.05, 0.95, 37)))

curve_rows: list[dict[str, float | bool]] = []
best: dict[str, float | bool] | None = None
for threshold in thresholds:
    for long_means_ai in (True, False):
        if long_means_ai:
            train_pred = (train_words >= threshold).astype(int)
            test_pred = (test_words >= threshold).astype(int)
        else:
            train_pred = (train_words <= threshold).astype(int)
            test_pred = (test_words <= threshold).astype(int)
        train_acc = float((train_pred == y_train).mean())
        test_acc = float((test_pred == y_test).mean())
        curve_rows.append(
            {
                "threshold": float(threshold),
                "long_means_ai": long_means_ai,
                "train_acc": train_acc,
                "test_acc": test_acc,
            }
        )
        if best is None or train_acc > float(best["train_acc"]):
            best = {
                "threshold": float(threshold),
                "long_means_ai": long_means_ai,
                "train_acc": train_acc,
                "test_acc": test_acc,
            }

assert best is not None
curve = pd.DataFrame(curve_rows)
chosen = curve.loc[curve["long_means_ai"] == bool(best["long_means_ai"])].sort_values("threshold")
majority_acc = float(max((y_test == 0).mean(), (y_test == 1).mean()))
direction = "AI if words >= threshold" if best["long_means_ai"] else "AI if words <= threshold"

fig, ax = plt.subplots(figsize=(7.4, 4.4))
ax.plot(chosen["threshold"], chosen["train_acc"], label="Train", color="#4C78A8")
ax.plot(chosen["threshold"], chosen["test_acc"], label="Test", color="#F58518", linestyle="--")
ax.axhline(majority_acc, color="#333333", linestyle=":", label=f"Test majority ({majority_acc:.1%})")
ax.axvline(float(best["threshold"]), color="#54A24B", linestyle="-.", label="Chosen on train")
ax.set_xlabel("Word-count threshold")
ax.set_ylabel("Accuracy")
ax.set_title(f"Length-only rule: {direction}")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, "length_only_baseline.png")
plt.show()

lift = float(best["test_acc"]) - majority_acc
print(f"cross-label duplicate texts: {n_conflicts:,}")
print(f"exact overlap — val/train: {val_exact:,}, test/train: {test_exact:,}, val/test: {val_test_exact:,}")
print(f"AI texts whose first 15 words also open a human text: {ai_prefix_rate:.1%}")
print(f"test texts whose first 15 words also open a train text: {test_prefix_rate:.1%}")
print(
    f"length-only test accuracy: {float(best['test_acc']):.1%} "
    f"at {float(best['threshold']):.0f} words ({direction}); "
    f"train accuracy {float(best['train_acc']):.1%}; majority baseline {majority_acc:.1%}"
)

if n_conflicts > 0:
    print(
        f"Verdict: {n_conflicts:,} strings carry both labels. "
        "Cleaning keeps the first copy, but those rows are noisy supervision."
    )
else:
    print("Verdict: no exact string is labeled both human and AI.")

if test_exact or val_exact or val_test_exact:
    print("Verdict: exact text crosses the split. That is leakage, and the split should be rebuilt after a stricter dedupe.")
else:
    print("Verdict: no exact train text is repeated in val or test.")

if test_prefix_rate >= 0.05 or ai_prefix_rate >= 0.05:
    print(
        "Verdict: shared openings are common enough that a model can memorize prompt templates "
        "instead of a general difference between human and AI writing."
    )
else:
    print("Verdict: shared 15-word openings are uncommon, so exact prompt reuse looks limited.")

if lift >= 0.10:
    print(
        "Verdict: length beats the majority baseline by at least 10 points on the test split. "
        "Put this length-only accuracy next to every model score before treating a gain as stylistic detection."
    )
elif lift >= 0.03:
    print(
        "Verdict: length is a real but modest cue. It should not dominate the error analysis, "
        "but it is large enough to report beside the model scores."
    )
else:
    print(
        "Verdict: length stays close to the majority baseline, so the classes are not trivially separable by word count."
    )


The threshold in the chart was picked by training accuracy alone; the dashed line is that same rule on the test split, not a second search. Exact overlap should be zero after duplicate removal. The opening check is the one that can still fire: many rows in this corpus are essays written to a shared prompt, and the first 15 words often *are* that prompt. A high rate means in-domain accuracy can be prompt memorization. A length rule that clears the majority baseline by a wide margin means the same thing for a different shortcut. Either result is a reason to keep the TF-IDF and DistilBERT numbers next to these two baselines, and to test both models on text from a different source.
